# Notebook 07 — Structure Comparison (RMSD) and Drug Design Survey

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 07 of 08  
**Time estimate:** 70 minutes

> RMSD is the standard metric for comparing protein structures.
> It is also the basis for evaluating AlphaFold2 and other prediction methods.
> Structure-based drug design extends the toolkit to the single most commercially
> important application of structural biology.

---
## Step 1 — Motivation

RMSD (Root Mean Square Deviation) appears in every structural biology paper:
benchmarks comparing predicted vs. experimental structures, MD simulation convergence,
clustering of conformational states. Implementing it correctly — including
the Kabsch superposition — is a concrete coding skill tested in interviews.

---
## Step 2 — Intuition

**RMSD without superposition:** If two structures are in different orientations,
RMSD is meaningless — it measures the distance between two different poses, not
the structural difference.

**Kabsch superposition:** Find the rotation matrix $R$ that minimizes the RMSD
between two sets of aligned atomic coordinates after translating both to their
centroids. The optimal $R$ is found via SVD.

**Structure-based drug design (SBDD):**  
If you know the 3D structure of a protein target (e.g. a kinase), you can
computationally screen thousands of small molecules to find those that fit the
binding site. This is *docking* — predicting the binding pose and affinity of
a ligand in a protein pocket.

---
## Step 3 — Biological Background

**Structural classification databases:**
- **SCOP2 (Structural Classification of Proteins 2):** Manual + automated hierarchy:
  Class → Fold → Superfamily → Family → Protein → Species
- **CATH:** Class, Architecture, Topology, Homologous superfamily — automated
- **ECOD:** Evolutionary Classification of Protein Domains — domain-centric

**Structure-based drug design pipeline:**
1. **Target identification:** which protein is causally involved in the disease?
2. **Structure determination:** X-ray/cryo-EM of the target (or AlphaFold2 prediction)
3. **Binding site identification:** geometric + conservation analysis (fpocket, SiteMap)
4. **Virtual screening:** dock a library of compounds (AutoDock Vina, Glide)
5. **Hit validation:** biochemical assays for binding affinity (Kd, IC50)
6. **Lead optimization:** medicinal chemistry to improve potency, selectivity, ADMET

**Fragment-based drug design:**  
Screen small fragments (MW < 300 Da) by NMR or X-ray. Fragments bind weakly (mM)
but efficiently. Link fragments that bind to adjacent pockets to generate drug-like
leads.

---
## Step 4 — Mathematical Explanation

**RMSD between two sets of $N$ aligned atoms:**
$$\text{RMSD} = \sqrt{\frac{1}{N} \sum_{i=1}^N \|\mathbf{r}_i^{A} - \mathbf{r}_i^{B}\|^2}$$

**Kabsch algorithm (optimal superposition):**

1. Translate both structures to their centroids:
   $P = A - \bar{A}$, $Q = B - \bar{B}$

2. Compute the covariance matrix:
   $H = P^T Q$

3. SVD: $H = U \Sigma V^T$

4. Optimal rotation:
   $R = V \begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & \det(VU^T) \end{pmatrix} U^T$

   (The $\det$ correction ensures a proper rotation, not a reflection.)

5. Apply rotation to $P$ and compute RMSD against $Q$.

**TM-score (Template Modelling score):**  
RMSD is sensitive to outliers (one misaligned loop inflates RMSD). TM-score
normalizes by protein length and down-weights large deviations:
$$\text{TM-score} = \frac{1}{L} \sum_i \frac{1}{1 + (d_i / d_0)^2}$$
where $d_0 = 1.24\sqrt[3]{L - 15} - 1.8$ Å and $L$ = target length.
TM-score > 0.5 = same fold; > 0.7 = similar topology.

In [ ]:
# Step 6 — Python: Kabsch RMSD from scratch
import numpy as np
import matplotlib.pyplot as plt

def kabsch_rmsd(A, B):
    """
    Compute the minimum RMSD between two sets of corresponding 3D points
    after optimal superposition (Kabsch algorithm).

    A, B: (N, 3) arrays of Cα coordinates
    Returns: (rmsd, R, t_A, t_B) where R is the optimal rotation matrix
    """
    A = np.array(A, dtype=float)
    B = np.array(B, dtype=float)
    assert A.shape == B.shape, 'A and B must have the same shape'
    N = A.shape[0]

    # 1. Translate to centroids
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    P = A - centroid_A
    Q = B - centroid_B

    # 2. Covariance matrix
    H = P.T @ Q   # (3, 3)

    # 3. SVD
    U, S, Vt = np.linalg.svd(H)

    # 4. Optimal rotation (with reflection correction)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T

    # 5. Rotate P and compute RMSD
    P_rotated = P @ R.T
    rmsd = np.sqrt(((P_rotated - Q)**2).sum(axis=1).mean())

    return rmsd, R, centroid_A, centroid_B

def tm_score(A, B):
    """
    TM-score between structures A and B (after Kabsch superposition).
    Uses the target length L = len(B).
    """
    L = len(B)
    d0 = 1.24 * (L - 15)**(1/3) - 1.8
    d0 = max(d0, 0.5)  # avoid negative d0 for short proteins
    _, R, cA, cB = kabsch_rmsd(A, B)
    P = (A - cA) @ R.T
    Q = B - cB
    d_i = np.linalg.norm(P - Q, axis=1)
    tm = (1 / L) * np.sum(1 / (1 + (d_i / d0)**2))
    return tm

rng = np.random.default_rng(42)

# ---- Simulate two structures: original + perturbed ----
# Original: 80-residue ideal helix
N_RES = 80
ca_orig = np.array([[2.3*np.cos(2*np.pi*i/3.6), 2.3*np.sin(2*np.pi*i/3.6), i*1.5]
                     for i in range(N_RES)])

# Perturbation 1: small (good prediction)
ca_small_noise = ca_orig + rng.normal(0, 0.5, ca_orig.shape)

# Perturbation 2: large (poor prediction)
ca_large_noise = ca_orig + rng.normal(0, 3.0, ca_orig.shape)

# Perturbation 3: same structure but randomly rotated and translated
theta = np.radians(37)
R_arb = np.array([[np.cos(theta), -np.sin(theta), 0],
                   [np.sin(theta),  np.cos(theta), 0],
                   [0,              0,             1]])
ca_rotated = (ca_orig @ R_arb.T) + np.array([10, -5, 3])

rmsd_small, _, _, _ = kabsch_rmsd(ca_small_noise, ca_orig)
rmsd_large, _, _, _ = kabsch_rmsd(ca_large_noise, ca_orig)
rmsd_rotated, _, _, _ = kabsch_rmsd(ca_rotated, ca_orig)
rmsd_rotated_naive = np.sqrt(((ca_rotated - ca_orig)**2).sum(axis=1).mean())

tms = tm_score(ca_small_noise, ca_orig)
tml = tm_score(ca_large_noise, ca_orig)

print('RMSD results:')
print(f'  Small noise (σ=0.5 ��):   RMSD = {rmsd_small:.2f} Å, TM-score = {tms:.3f}')
print(f'  Large noise (σ=3.0 Å):   RMSD = {rmsd_large:.2f} Å, TM-score = {tml:.3f}')
print(f'  Same structure, rotated:  RMSD (Kabsch) = {rmsd_rotated:.4f} Å')
print(f'  Same structure, rotated:  RMSD (naive, no superposition) = {rmsd_rotated_naive:.2f} Å')
print('  → Kabsch correctly recovers near-zero RMSD for the rotated copy')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Superimposed structures (2D projection)
ax = axes[0]
# Superpose small noise onto original
_, R_s, cA_s, cB_s = kabsch_rmsd(ca_small_noise, ca_orig)
ca_small_sup = (ca_small_noise - cA_s) @ R_s.T + cB_s

ax.plot(ca_orig[:, 0], ca_orig[:, 2], 'b-', lw=1.5, label='Original', alpha=0.8)
ax.plot(ca_small_sup[:, 0], ca_small_sup[:, 2], 'r--', lw=1.5,
         label=f'Prediction (RMSD={rmsd_small:.1f}Å)', alpha=0.8)
ax.set_xlabel('X (Å)')
ax.set_ylabel('Z (Å)')
ax.set_title('A. Structure superposition\n(Kabsch-aligned Cα traces)')
ax.legend(fontsize=8)

# Panel B: RMSD vs. noise level
ax = axes[1]
noise_levels = np.linspace(0.1, 5.0, 30)
rmsd_vals = []
tm_vals = []
for sigma in noise_levels:
    ca_noisy = ca_orig + rng.normal(0, sigma, ca_orig.shape)
    r, _, _, _ = kabsch_rmsd(ca_noisy, ca_orig)
    t = tm_score(ca_noisy, ca_orig)
    rmsd_vals.append(r)
    tm_vals.append(t)
ax.plot(noise_levels, rmsd_vals, 'b-', lw=2, label='RMSD (Å)')
ax2t = ax.twinx()
ax2t.plot(noise_levels, tm_vals, 'r--', lw=2, label='TM-score')
ax2t.axhline(0.5, color='red', ls=':', alpha=0.5)
ax2t.set_ylabel('TM-score', color='red')
ax.set_xlabel('Noise σ (Å)')
ax.set_ylabel('RMSD (Å)', color='blue')
ax.set_title('B. RMSD and TM-score vs. noise\n(TM>0.5 = same fold)')
ax.legend(loc='upper left', fontsize=8)
ax2t.legend(loc='center right', fontsize=8)

# Panel C: Drug design pipeline overview
ax = axes[2]
ax.axis('off')
steps_dd = [
    ('1. Target', 'Identify protein; get structure (X-ray/AF2)',  0.88),
    ('2. Binding site', 'Cavity detection (fpocket, SiteMap)',   0.76),
    ('3. Virtual screen', 'Dock library; score binding poses',  0.64),
    ('4. Hit selection', 'Filter by score, ADMET, diversity',   0.52),
    ('5. Validation', 'Biochemical assay (IC50, Kd)',            0.40),
    ('6. Optimization', 'Medicinal chemistry; improve ADMET',   0.28),
    ('7. Lead', 'Candidate for preclinical development',        0.16),
]
for name, desc, y in steps_dd:
    ax.text(0.02, y, f'{name}:', fontsize=9, fontweight='bold', color='steelblue')
    ax.text(0.28, y, desc, fontsize=8)
    if y > 0.18:
        ax.annotate('', xy=(0.12, y - 0.10), xytext=(0.12, y - 0.05),
                      arrowprops={'arrowstyle': '->', 'color': 'grey', 'lw': 1.2})
ax.set_title('C. Structure-based drug design pipeline', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.suptitle('Module 11 NB07: Structure Comparison and Drug Design Survey', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('structure_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `kabsch_rmsd(A, B)` from scratch. Verify that for two identical
   structures in different orientations, RMSD = 0.
2. RMSD is sensitive to outliers. Demonstrate this by adding one residue with
   a very large error (say, 20 Å) to an otherwise near-perfect prediction.
   How does it affect RMSD vs. TM-score?
3. What is the difference between RMSD and TM-score? When would you prefer one
   over the other?
4. (Survey) What is molecular docking? What does a docking score represent?

---
## Step 10 — Quiz

1. Write the formula for RMSD. What are its units?
2. Why is superposition (Kabsch algorithm) necessary before computing RMSD?
3. What SVD step is the core of the Kabsch algorithm?
4. What does TM-score > 0.5 indicate?
5. What is the key difference between homology modelling and AlphaFold2?

---
## Step 12 — Reflection

> *[In one paragraph: a team wants to use structure-based drug design to target
> an intrinsically disordered protein (IDP). Explain why this is challenging,
> what pLDDT would tell you about an AlphaFold2 prediction of the IDP, and
> what experimental approach might still be useful.]*

---
*Next: `08_module_assessment.ipynb`*